# VERITAS 06 — Temporal knowledge + evidence graph + verification

**Phases 7–10.** This is where VERITAS stops being a RAG system.

## The failure being fixed

Two sources say different things. There are two very different reasons:

* **(a) a source is WRONG** — it contradicts reality
* **(b) a source WAS RIGHT, in 2023** — it contradicts only the present

A vector index cannot tell these apart: both chunks mention "CEO", both rank
highly, the model averages them into a confident wrong answer. Distinguishing
them requires storing *when a fact was true*.

## Bitemporal storage (Snodgrass, 1995)

Two independent time axes:

* **valid time** `[valid_from, valid_to)` — when it was true **in the world**
* **transaction time** `[recorded_at, superseded_at)` — when **we knew** it

They genuinely come apart: a filing published March 2026 states a CEO change
effective January 2026. With one axis you must choose which lie to tell. With
both, all four questions are answerable — including *"who did we think was CEO
in 2024, back in 2024?"*, which is the audit trail.

**Nothing is deleted.** A change closes valid time and opens a new interval; a
correction closes transaction time and keeps valid time. Different operations,
both preserving history.

As-of lookups are a **binary search** over versions sorted by `valid_from`:
`O(log n)`, not a scan.

## Evidence graph: why a graph, not a table

The questions are *path* questions. The decisive one:
**are these three sources independent?** Three outlets rewriting one wire story
look like three-source corroboration in a table. In a graph they converge on
one `DERIVED_FROM` ancestor, so the corroboration bonus is correctly withheld.
Independence is a **topological** property.

## Verification is entailment, not similarity

"Acme did **not** appoint Y" has ~0.95 cosine similarity to "Acme appointed Y".
Retrieval scores are useless here. Two channels:

1. **symbolic** — numeric mismatch (after unit normalisation), polarity flip,
   temporal mismatch. Auditable, ~free, no hallucination mode.
2. **neural NLI** — paraphrase, which no rule catches.

Symbolic runs first and can **veto** the model. A claim is SUPPORTED only when
no symbolic check fires *and* entailment clears threshold.

In [ ]:
import sys, os, time, math, json, random
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, torch
torch.manual_seed(1337); np.random.seed(1337); random.seed(1337)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE, '| torch', torch.__version__)
if DEVICE == 'cuda':
    print('gpu:', torch.cuda.get_device_name(0),
          f'| {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
from veritas.temporal.versioning import TemporalStore, ChangeKind, now_utc
from veritas.temporal.change_detection import ChangeDetector, simhash, hamming, content_digest
from veritas.temporal.temporal_retrieval import (parse_temporal_query, TemporalIntent,
        freshness_score, interval_overlap_score, make_signal_fns, HALFLIFE_DAYS)
from veritas.evidence.claims import extract_claims, claims_to_state
from veritas.evidence.graph import EvidenceGraph, EdgeType
from veritas.evidence.quality import SourcePolicy, score_evidence, support_label, explain
from veritas.evidence.verifier import ClaimVerifier, EvidenceItem, Verdict, lexical_entailment
from veritas.evidence.contradiction import ContradictionDetector, ConflictType

## 1. The bitemporal store — all four question types

In [ ]:
store = TemporalStore()
store.alias('acme', 'Acme Industries')   # entity resolution: without it, two timelines

store.assert_fact('Acme Industries','ceo','Dana Whitfield', valid_from='2024-01-01',
                  recorded_at='2024-03-01', source_id='ir.acme.com', confidence=0.8)
store.assert_fact('Acme Industries','ceo','Priya Raman', valid_from='2025-07-01',
                  recorded_at='2025-06-15', source_id='ir.acme.com', confidence=0.8)
v, ev = store.assert_fact('Acme Industries','ceo','Marcus Lund', valid_from='2026-02-01',
                  recorded_at='2026-02-02', source_id='sec.gov', confidence=0.9)
print('change kind:', ev.kind, '| previous:', ev.old_value)

print('\nQ1 who is CEO now              ->', store.current('Acme Industries','ceo').value)
print('Q2 who was CEO in 2024         ->', store.as_of('Acme Industries','ceo','2024-06-01').value)
print('Q3 what did we believe in 2025 ->', store.as_of("Acme Industries",'ceo','2025-09-01',
                                                       known_at='2025-09-01').value)
print('Q4 when did we learn of Lund   ->', v.recorded_at.date(),
      f'(effective {v.valid_from.date()} — {(v.recorded_at-v.valid_from).days:+d} days)')
print('\nOUTDATED (a naive retriever would quote these as fact):',
      [x.value for x in store.outdated('Acme Industries','ceo')])

In [ ]:
print('full timeline:')
for x in store.timeline('Acme Industries'):
    end = 'present' if x.valid_to.year > 9000 else x.valid_to.date()
    print(f'  {x.valid_from.date()} → {end:>10}  {x.attribute}={x.value:16s} '
          f'[{x.change_kind:10s}] src={x.source_id} recorded={x.recorded_at.date()}')

# A restatement by a different source is corroboration, not a new fact:
before = store.current('Acme Industries','ceo').confidence
store.assert_fact('Acme Industries','ceo','Marcus Lund', valid_from='2026-02-01',
                  source_id='reuters.com', recorded_at='2026-02-05')
print(f'\nREAFFIRMED by a second source: confidence {before:.2f} -> '
      f'{store.current("Acme Industries","ceo").confidence:.2f} (no duplicate row written)')

## 2. Change detection — a cascade, cheapest filter first

Re-ingesting a source costs a parse + claim extraction + embedding + index
write. Most polls return an identical page, or one whose only difference is a
rotating banner. A system that reprocesses everything cannot run continuously.

**SimHash** projects a document to a 64-bit signature where similar documents
have small Hamming distance — comparison is one `popcount(a^b)` instruction.
Stage 4 is the one that matters: text changing is not news, a **claim**
changing is.

In [ ]:
det = ChangeDetector()
base = 'Acme Industries filing. The chief executive is Priya Raman. Revenue was 1.27 billion euros.'
tests = [
    ('first sight',        base),
    ('byte-identical',     base),
    ('cosmetic banner',    base + '\nLast updated 14:03:22'),
    ('reworded, same fact',base.replace('The chief executive is','Chief executive:')),
    ('REAL state change',  base.replace('Priya Raman','Marcus Lund')),
]
for label, text in tests:
    claims = extract_claims(text, 'd','d','Acme Industries')
    r = det.check('acme_ir', text, claims_to_state(claims))
    print(f'{label:22s} changed={str(r.changed):5s} reason={r.reason:16s} sim={r.similarity:.3f} '
          f'state-change={r.has_state_change}')
print(f'\npolls={det.state("acme_ir").poll_count} changes={det.state("acme_ir").change_count} '
      f'volatility={det.state("acme_ir").volatility:.2f}')
print(f'adaptive next poll: {det.next_poll_seconds("acme_ir")}s '
      f'(stable sources back off, volatile ones speed up)')

## 3. Temporal query understanding

The retriever must know the **tense** of the question. Note that CHANGE
deliberately switches *off* freshness weighting — a change question needs the
before *and* the after, so preferring recent evidence would destroy the answer.

In [ ]:
for q, attr in [('Who is the current CEO?','ceo'),
                ('Who was the CEO in 2022?','ceo'),
                ('As of March 2025, what was the status?','status'),
                ('How did the CEO change over time?','ceo'),
                ('What happened between 2020 and 2024?','ceo'),
                ('What is the share price?','price')]:
    tq = parse_temporal_query(q, attr)
    print(f'{q:44s} -> {tq.intent:11s} anchor={tq.anchor.date()} '
          f'half-life={tq.halflife_days:>6.1f}d freshness={tq.apply_freshness}')
print('\nHalf-life is PER-ATTRIBUTE. One global decay is the usual mistake:')
print('it would decay a founding date at the same rate as a share price.')

In [ ]:
import matplotlib.pyplot as plt
ages = np.linspace(0, 1095, 200)
plt.figure(figsize=(7,3))
for attr in ['price','status','revenue','ceo','founded']:
    hl = HALFLIFE_DAYS[attr]
    plt.plot(ages, 0.5**(ages/hl), label=f'{attr} (t½={hl:g}d)')
plt.xlabel('document age (days)'); plt.ylabel('freshness'); plt.legend(fontsize=8)
plt.title('exponential decay, not a cliff — a 366-day cutoff would drop the only evidence there is')
plt.grid(alpha=.3); plt.show()

## 4. Evidence graph and the independence test

This is the cell that separates VERITAS from a citation-listing RAG.

In [ ]:
g = EvidenceGraph()
g.add_source('reuters.com','Reuters',tier=2,domain='corporate')
g.add_source('dailyfeed.example.com','Daily Feed',tier=3)
g.add_source('aggregator.example.net','Aggregator',tier=3)
g.add_source('gov.example.gov','Regulator',tier=1)

g.add_document('wire_orig','reuters.com',date='2026-03-03')
for rid, src in [('reprint_a','dailyfeed.example.com'), ('reprint_b','aggregator.example.net')]:
    g.add_document(rid, src, date='2026-03-04')
    g.link(f'doc:{rid}', 'doc:wire_orig', EdgeType.DERIVED_FROM)   # syndication edge
g.add_document('filing','gov.example.gov',date='2026-03-10')

claim = g.add_claim('c1','Helios Energy has begun construction at Almeria',
                    entity='Helios Energy', attribute='status', value='construction',
                    doc_id='wire_orig', valid_from='2026-03-03')
for d in ['reprint_a','reprint_b','filing']:
    g.supports(f'doc:{d}', claim, weight=0.9)

print(f'documents citing this claim : {len(g.evidence_for(claim))}')
print(f'INDEPENDENT sources         : {g.independent_sources(claim)}')
print('\nFour documents, two independent roots. A flat (claim, source) table would')
print('report four-source corroboration for what is really a wire story plus one filing —')
print('and that inflated count is how a system talks itself into a false fact.')
print('\nfirst reported by:', g.first_reported(claim))

In [ ]:
import json as _j
print(_j.dumps(g.provenance_path(claim), indent=1)[:900])
print('\ngraph:', g.stats())

## 5. Evidence quality — a support LABEL, never a truth percentage

Seven bounded factors, weighted, with a **hard floor**: zero independent
sources ⇒ score 0, regardless of how well everything else scores. A plain
weighted sum would let five weak signals outvote the absence of evidence.

In [ ]:
policy = SourcePolicy()
print('Source tiers are PER-DOMAIN — there is no universal ranking:')
for src in ['sec.gov','gov.example.gov','reuters.com','status.acme.com','arxiv.org','reddit.com']:
    tiers = {d: policy.tier(src, d) for d in ['government','corporate','science','software']}
    print(f'  {src:22s} {tiers}')
print('\nFor a regulatory question the regulator outranks Reuters; for a software')
print('outage the vendor status page outranks the regulator. Same source, different tier.')

In [ ]:
scenarios = [
 ('one tier-1 primary, fresh',      ['sec.gov'], ['2026-08-01'], 1, 1.0),
 ('two independent tier-1, fresh',  ['sec.gov','gov.example.gov'], ['2026-08-01','2026-08-03'], 2, 1.0),
 ('two sources, disagreeing',       ['sec.gov','reuters.com'], ['2026-08-01','2026-08-02'], 2, 0.5),
 ('one tier-4 blog, 3 years old',   ['reddit.com'], ['2023-01-01'], 1, 1.0),
 ('NO independent source',          ['sec.gov'], ['2026-08-01'], 0, 1.0),
]
for name, srcs, dates, n_ind, agree in scenarios:
    s, f = score_evidence(sources=srcs, dates=dates,
        texts=['Acme reported revenue of 1.42 billion euros on 2026-08-01.'],
        n_independent=n_ind, agreement=agree, validity=1.0, domain='corporate',
        ref_time=__import__('datetime').datetime(2026,9,1,tzinfo=__import__('datetime').timezone.utc))
    lab = support_label(s, n_ind, has_conflict=agree < 0.6)
    print(f'{name:32s} {explain(s, f, lab)}')
print('\nNote the last row: the hard floor. No corroboration -> INSUFFICIENT, whatever else says.')

## 6. Claim verification — the symbolic checks that catch real errors

In [ ]:
ver = ClaimVerifier()
def check(claim_text, evidence_text, src='reuters.com', date='2026-08-14', valid_to=None):
    c = extract_claims(claim_text, 'gen','gen','Nova Logistics')[0]
    e = [EvidenceItem('d1', evidence_text, src, date, 2, valid_to=valid_to)]
    v = ver.verify_claim(c, e, anchor_time='2026-09-01')
    print(f'{v.verdict:18s} entail={v.entailment:.2f}  {v.reason}')
    return v

print('NUMERIC MISMATCH — the highest-yield check (fabricated numbers are the most')
print('common and most damaging RAG error, and cosine scores the two as near-identical):')
check('Nova Logistics opened 15 offices in India in 2026.',
      'Nova Logistics opened 12 offices in India in 2026.')
print('\nUNIT NORMALISATION — 1.4 billion == 1,400 million, NOT a conflict:')
check('Nova Logistics reported revenue of 1.4 billion.',
      'Nova Logistics reported revenue of 1,400 million.')
print('\nPOLARITY FLIP:')
check('Nova Logistics opened 12 offices in India in 2026.',
      'Nova Logistics did not open offices in India in 2026.')
print('\nCLEAN SUPPORT:')
check('Nova Logistics opened 12 offices in India in 2026.',
      'Nova Logistics opened 12 new offices across India during 2026, filings show.')
print('\nTEMPORAL MISMATCH — true earlier, not now. NOT the same as "refuted":')
check('Nova Logistics operates 12 offices in India.',
      'Nova Logistics operated 12 offices in India.', date='2023-01-01', valid_to='2024-01-01')

## 7. Contradiction analysis — four kinds, only one is a real conflict

In [ ]:
cd = ContradictionDetector()
h = store.history('Acme Industries','ceo')
c = cd.compare_versions(h[0], h[-1], 'corporate')
print(f'[{c.type}] sev={c.severity:.1f}  {c.explanation}')
print('   ^ successive states. Reporting this as a disagreement would fire on')
print('     every entity that ever changed — which is how a conflict detector becomes noise.\n')

s2 = TemporalStore()
a,_ = s2.assert_fact('Nova Logistics','offices','15', valid_from='2026-01-01',
                     valid_to='2026-12-31', source_id='ir.novalogistics.com')
b,_ = s2.assert_fact('Nova Logistics','offices','12', valid_from='2026-01-01',
                     valid_to='2026-12-31', source_id='reddit.com')
c2 = cd.compare_versions(a, b, 'corporate')
print(f'[{c2.type}] sev={c2.severity:.1f}  {c2.explanation}')
print(f'   preference: {c2.preference_reason or "none — reported unresolved"}')
print('\n' + cd.summarize([c, c2]))

Next: **07 — the agentic loop, end to end.**